## Antenna traffic single-step evaluation (Chronos)
We evaluate Chronos models on histo_trafic_....csv files by forecasting the last time step for each antenna sector using a fixed 128-point context for all models.
The helper scripts `dataprocessing.py` and `single_eval.py` handle preprocessing and RMSE scoring (absolute and percentage), averaging 10 random one-step samples per sector. The cells below run on device; edit `MODEL_IDS` to try more Chronos variants later and 'DATASET' to select the right dataset.jsonl to work on.


In [ ]:
import pandas as pd

from scripts.utils.notebook_helpers import (
    RESULTS_DIR,
    device,
    ensure_processed_data,
    evaluate_model,
)

DATASET = "instant"  # choose among: "original", "instant", "instant_short"
print("Using device:", device)
print("Using dataset:", DATASET)

ensure_processed_data(DATASET)



Using device: cpu
Using dataset: instant


PosixPath('data/processed_trafic_instant.jsonl')

In [17]:
MODEL_IDS = [
    "amazon/chronos-2",
    "amazon/chronos-bolt-tiny",
    "amazon/chronos-bolt-mini",
    "amazon/chronos-bolt-small",
    "amazon/chronos-bolt-base",
    "amazon/chronos-t5-tiny",
    "amazon/chronos-t5-mini",
    "amazon/chronos-t5-small",
    "amazon/chronos-t5-base",
    "amazon/chronos-t5-large",
]
NUM_SAMPLES = 8  # tweak for slower/fast CPU evaluations
CONTEXT_LENGTH = 128  # enforce uniform context length across models
RMSE_SAMPLES = 20  # number of random windows per sector to average RMSE metrics


In [18]:
results = [
    evaluate_model(
        mid,
        num_samples=NUM_SAMPLES,
        dataset=DATASET,
        context_length=CONTEXT_LENGTH,
        rmse_samples=RMSE_SAMPLES,
    )
    for mid in MODEL_IDS
]

summary_df = pd.DataFrame(
    {
        "model_id": [r["model_id"] for r in results],
        "rmse_percentage": [r["rmse_percentage"] for r in results],
        "rmse": [r["rmse"] for r in results],
        "rmse_ratio": [r["rmse_ratio"] for r in results],
        "num_series": [r["num_series"] for r in results],
        "num_samples": [r["num_samples"] for r in results],
        "context_length": [r["context_length"] for r in results],
        "rmse_samples": [r["rmse_samples"] for r in results],
        "skipped_sectors": [len(r.get("skipped_sectors", [])) for r in results],
    }
)
summary_df


Evaluating on 'instant': /Users/colinminini/Desktop/SCOC_ICE/LLM6G-Repository/.venv/bin/python scripts/single_eval.py --model-id amazon/chronos-2 --data-path data/processed_trafic_instant.jsonl --num-samples 8 --context-length 128 --rmse-samples 20 --device cpu --output-path results/evals/eval_amazon__chronos-2_instant.json


/Users/colinminini/Desktop/SCOC_ICE/LLM6G-Repository/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Model amazon/chronos-2 | RMSE%=55.8018% (RMSE=2.6367) across 86 sectors (20 samples/sector, context=128).
Saved detailed results to results/evals/eval_amazon__chronos-2_instant.json
Evaluating on 'instant': /Users/colinminini/Desktop/SCOC_ICE/LLM6G-Repository/.venv/bin/python scripts/single_eval.py --model-id amazon/chronos-bolt-tiny --data-path data/processed_trafic_instant.jsonl --num-samples 8 --context-length 128 --rmse-samples 20 --device cpu --output-path results/evals/eval_amazon__chronos-bolt-tiny_instant.json
Model amazon/chronos-bolt-tiny | RMSE%=44.4600% (RMSE=2.8548) across 86 sectors (20 samples/sector, context=128).
Saved detailed results to results/evals/eval_amazon__chronos-bolt-tiny_instant.json
Evaluating on 'instant': /Users/colinminini/Desktop/SCOC_ICE/LLM6G-Repository/.venv/bin/python scripts/single_eval.py --model-id amazon/chronos-bolt-mini --data-path data/processed_trafic_instant.jsonl --num-samples 8 --context-length 128 --rmse-samples 20 --device cpu --output

,model_id,rmse_percentage,rmse,rmse_ratio,num_series,num_samples,context_length,rmse_samples,skipped_sectors
0,amazon/chronos-2,55.801753,2.636683,0.558018,86,8,128,20,0
1,amazon/chronos-bolt-tiny,44.460049,2.854801,0.444600,86,8,128,20,0
2,amazon/chronos-bolt-mini,43.983069,2.873875,0.439831,86,8,128,20,0
3,amazon/chronos-bolt-small,43.237556,2.888732,0.432376,86,8,128,20,0
4,amazon/chronos-bolt-base,43.702037,2.876326,0.437020,86,8,128,20,0
5,amazon/chronos-t5-tiny,45.183892,2.995239,0.451839,86,8,128,20,0
6,amazon/chronos-t5-mini,45.157872,3.007770,0.451579,86,8,128,20,0
7,amazon/chronos-t5-small,45.324630,3.008382,0.453246,86,8,128,20,0
8,amazon/chronos-t5-base,45.345237,3.045875,0.453452,86,8,128,20,0
9,amazon/chronos-t5-large,44.324371,3.027236,0.443244,86,8,128,20,0


In [19]:
PARAM_COUNTS = {
    "amazon/chronos-2": "120M",
    "amazon/chronos-bolt-tiny": "9M",
    "amazon/chronos-bolt-mini": "21M",
    "amazon/chronos-bolt-small": "48M",
    "amazon/chronos-bolt-base": "205M",
    "amazon/chronos-t5-tiny": "8M",
    "amazon/chronos-t5-mini": "20M",
    "amazon/chronos-t5-small": "46M",
    "amazon/chronos-t5-base": "200M",
    "amazon/chronos-t5-large": "710M",
}

order = MODEL_IDS
summary_ranked = summary_df.copy()
summary_ranked["parameters"] = summary_ranked["model_id"].map(PARAM_COUNTS)
summary_ranked["model_id"] = pd.Categorical(
    summary_ranked["model_id"], categories=order, ordered=True
)
summary_ranked = summary_ranked.sort_values("model_id")[
    [
        "model_id",
        "rmse_percentage",
        "rmse",
        "parameters",
        "rmse_ratio",
        "num_series",
        "num_samples",
        "context_length",
        "rmse_samples",
        "skipped_sectors",
    ]
]

# Save the ranked summary to a CSV file
summary_ranked.to_csv(RESULTS_DIR / f"model_performance_summary_{DATASET}.csv", index=False)
summary_ranked


,model_id,rmse_percentage,rmse,parameters,rmse_ratio,num_series,num_samples,context_length,rmse_samples,skipped_sectors
0,amazon/chronos-2,55.801753,2.636683,120M,0.558018,86,8,128,20,0
1,amazon/chronos-bolt-tiny,44.460049,2.854801,9M,0.444600,86,8,128,20,0
2,amazon/chronos-bolt-mini,43.983069,2.873875,21M,0.439831,86,8,128,20,0
3,amazon/chronos-bolt-small,43.237556,2.888732,48M,0.432376,86,8,128,20,0
4,amazon/chronos-bolt-base,43.702037,2.876326,205M,0.437020,86,8,128,20,0
5,amazon/chronos-t5-tiny,45.183892,2.995239,8M,0.451839,86,8,128,20,0
6,amazon/chronos-t5-mini,45.157872,3.007770,20M,0.451579,86,8,128,20,0
7,amazon/chronos-t5-small,45.324630,3.008382,46M,0.453246,86,8,128,20,0
8,amazon/chronos-t5-base,45.345237,3.045875,200M,0.453452,86,8,128,20,0
9,amazon/chronos-t5-large,44.324371,3.027236,710M,0.443244,86,8,128,20,0
